# Practical 6: Facial Emotion Recognition using a CNN (CK+ Dataset)

**Aim:** To classify facial emotions on the lab-controlled CK+48 dataset (981 images, 48 x 48 grayscale) using a CNN, and compare the result with FER-2013.

> Reconstructed code (the source PDF has no code). Not executed by the author - run cells in order (Kernel -> Restart & Run All) and check dataset paths in the first cells. Expected values: Test accuracy ~91% (CK+ is lab-controlled, posed apex expressions).; Only 981 images -> 80/20 split. Subject-independent split is ideal but not enforced.

## Cell 1 - Import libraries

**What it does:**
- `train_test_split` will create the 80/20 train/test split.

*Expected output (illustrative):* TensorFlow version: 2.x.x

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, classification_report, ConfusionMatrixDisplay

print("TensorFlow version:", tf.__version__)

## Cell 2 - Load the images into NumPy arrays

**What it does:**
- Reads every image from the class folders in a fixed order (`shuffle=False`) and stacks them into one array `X` (images) and `y` (integer labels).
- The dataset is tiny (981 images), so it comfortably fits in memory.

*Expected output (illustrative):* Classes: ['anger', 'contempt', 'disgust', 'fear', 'happy', 'sadness', 'surprise']
X: (981, 48, 48, 1) y: (981,)

In [ ]:
DATA_DIR = "CK+48"

ds = keras.utils.image_dataset_from_directory(
    DATA_DIR, color_mode="grayscale", image_size=(48, 48),
    batch_size=64, shuffle=False)

class_names = ds.class_names
X = np.concatenate([x.numpy() for x, _ in ds])
y = np.concatenate([lab.numpy() for _, lab in ds])

print("Classes:", class_names)
print("X:", X.shape, "y:", y.shape)
print("Images per class:", np.bincount(y))

## Cell 3 - Normalise and split into train / test (80/20)

**What it does:**
- Pixels are scaled to [0,1].
- `test_size=0.2`: 80% training (784 images), 20% test (197 images) as stated in your PDF.
- `stratify=y` keeps the class proportions the same in both sets; `random_state=42` makes the split repeatable.
- Note: this is a random image-level split, not subject-independent, so accuracy may be slightly optimistic.

*Expected output (illustrative):* Train: (784, 48, 48, 1)  Test: (197, 48, 48, 1)

In [ ]:
X = X.astype("float32") / 255.0

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

print("Train:", x_train.shape, " Test:", x_test.shape)

## Cell 4 - Show sample faces

**What it does:**
- Quick visual check that images and labels match.

*Expected output (illustrative):* 10 clear frontal faces with emotion names.

In [ ]:
plt.figure(figsize=(10, 4))
for i in range(10):
    plt.subplot(2, 5, i + 1)
    plt.imshow(x_train[i].squeeze(), cmap="gray")
    plt.title(class_names[y_train[i]])
    plt.axis("off")
plt.tight_layout()
plt.show()

## Cell 5 - Build the CNN

**What it does:**
- Architecture from your PDF: Conv2D(32 -> 64 -> 128) -> MaxPooling -> Flatten -> Dense(128) -> Dense(7, softmax).
- With no padding, sizes shrink: 48 -> 46 -> 23 -> 21 -> 10 -> 8 -> 4, so Flatten gives 4 x 4 x 128 = 2,048 values.
- The last layer size comes from `len(class_names)` = 7.

*Expected output (illustrative):* Output shapes: (46,46,32) (23,23,32) (21,21,64) (10,10,64) (8,8,128) (4,4,128) (2048) (128) (7). Total params: 355,847.

In [ ]:
model = keras.Sequential([
    keras.Input(shape=(48, 48, 1)),
    layers.Conv2D(32, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(64, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Conv2D(128, (3, 3), activation="relu"),
    layers.MaxPooling2D((2, 2)),
    layers.Flatten(),
    layers.Dense(128, activation="relu"),
    layers.Dense(len(class_names), activation="softmax"),
])

model.summary()

## Cell 6 - Compile and train

**What it does:**
- Same loss/optimizer as before. `batch_size=32` suits a small dataset.
- `validation_split=0.1` takes 10% of the training arrays for validation. (Keras takes the *last* 10% of the array, so the training data was shuffled by `train_test_split` already.)
- EarlyStopping avoids overfitting the small dataset.

*Expected output (illustrative):* Training accuracy quickly approaches ~0.95-1.0; validation accuracy high (~0.9).

In [ ]:
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=0.001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=8, restore_best_weights=True)

history = model.fit(
    x_train, y_train,
    epochs=30,
    batch_size=32,
    validation_split=0.1,
    callbacks=[early_stop],
)

## Cell 7 - Accuracy and loss curves

**What it does:**
- Graphs show learning progress; on CK+ the curves rise much faster than on FER-2013.

*Expected output (illustrative):* Two graphs; accuracy reaches high values within ~10 epochs.

In [ ]:
plt.figure(figsize=(11, 4))

plt.subplot(1, 2, 1)
plt.plot(history.history["accuracy"], label="Train")
plt.plot(history.history["val_accuracy"], label="Validation")
plt.title("Accuracy")
plt.xlabel("Epoch")
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(history.history["loss"], label="Train")
plt.plot(history.history["val_loss"], label="Validation")
plt.title("Loss")
plt.xlabel("Epoch")
plt.legend()

plt.tight_layout()
plt.show()

## Cell 8 - Test accuracy, confusion matrix, precision/recall/F1

**What it does:**
- Evaluates on the 197 held-out images and prints per-class metrics.

*Expected output (illustrative):* Test accuracy about 0.91 (from your PDF: ~91%). A mostly diagonal 7 x 7 matrix.

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print("Test accuracy:", round(test_acc, 4))

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)

cm = confusion_matrix(y_test, y_pred)
ConfusionMatrixDisplay(cm, display_labels=class_names).plot(cmap="Blues", xticks_rotation=45)
plt.title("Confusion Matrix - CK+")
plt.show()

print(classification_report(y_test, y_pred, target_names=class_names, digits=3))

## Cell 9 - Predict sample images and save the model

**What it does:**
- Displays random test faces with true/predicted labels; most should be green.

*Expected output (illustrative):* 8 faces with titles, mostly green.

In [ ]:
idx = np.random.choice(len(x_test), 8, replace=False)

plt.figure(figsize=(12, 6))
for pos, i in enumerate(idx):
    plt.subplot(2, 4, pos + 1)
    plt.imshow(x_test[i].squeeze(), cmap="gray")
    ok = y_pred[i] == y_test[i]
    plt.title("True: " + class_names[y_test[i]] + "\nPred: " + class_names[y_pred[i]],
              color="green" if ok else "red", fontsize=9)
    plt.axis("off")
plt.tight_layout()
plt.show()

model.save("ckplus_cnn.keras")

## Result / Conclusion

A simple 3-block CNN trained on CK+48 reached ~91% test accuracy, much higher than on FER-2013, because CK+ is clean and lab-controlled.